In [1]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [2]:
"""
=============================================================================
IoT-Based Synthetic Traces Dataset Generator
Paper: "Adaptive Data Mining Framework for Decentralized Backup and
        Retrieval in Edge-Cloud Environments" (ADM-DE)

"""

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random
import os

# ── Mount Google Drive ───────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("✅ Google Drive mounted successfully.")
except ImportError:
    print("⚠️  Not running in Colab – skipping Drive mount.")
except Exception as e:
    print(f"⚠️  Drive mount issue: {e}")

# ── Output Path ──────────────────────────────────────────────────────────────
OUTPUT_DIR  = "/content/drive/MyDrive/Github/Parthiban"
OUTPUT_FILE = "iot_synthetic_traces.csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"📁 Save path : {OUTPUT_DIR}/{OUTPUT_FILE}\n")

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# ── Parameters (from paper) ──────────────────────────────────────────────────
NUM_EDGE_NODES      = 50
NUM_CLOUD_GATEWAYS  = 5
LATENCY_MIN_MS      = 5
LATENCY_MAX_MS      = 120
STORAGE_MIN_TB      = 1
STORAGE_MAX_TB      = 10
FAILURE_PROB_MIN    = 0.05
FAILURE_PROB_MAX    = 0.15
NUM_DATA_ITEMS      = 500
NUM_RECORDS         = 10_000

# SOM grid (paper section 3.2)
SOM_GRID = (20, 20)

# Priority weight coefficients β1, β2, β3 (paper eq. 1)
BETA1, BETA2, BETA3 = 0.6, 0.25, 0.15

DATA_TYPES = [
    "sensor_reading", "video_stream", "log_file",
    "telemetry", "image_frame", "config_update", "event_alert"
]

# ─────────────────────────────────────────────────────────────────────────────
# Step 1 – Generate Raw IoT Synthetic Traces
# ─────────────────────────────────────────────────────────────────────────────

def generate_iot_synthetic_traces(n_records: int = NUM_RECORDS) -> pd.DataFrame:
    """
    Generates IoT-based synthetic traces as described in paper section 3.1.

    Distributions:
      access_frequency    → Poisson  (λ=5)   — sporadic, independent events
      latency_ms          → Log-normal       — most values low, tail for congestion
      recovery_time_ms    → Log-normal       — most values low, tail for failures
      bandwidth_mb_s      → Uniform          — heterogeneous node bandwidth
      storage_overhead_pct→ Uniform          — heterogeneous node storage
    """
    print(f"[Step 1] Generating {n_records:,} raw IoT trace records …")

    base_time = datetime(2024, 1, 1)
    node_ids  = [f"edge_node_{i:02d}" for i in range(1, NUM_EDGE_NODES + 1)]
    data_ids  = [f"data_{i:04d}"      for i in range(1, NUM_DATA_ITEMS + 1)]

    # Timestamps spread across 30 days
    timestamps = sorted([
        base_time + timedelta(seconds=int(t))
        for t in np.random.uniform(0, 30 * 86_400, n_records)
    ])

    df = pd.DataFrame({
        # ── Identity features ──────────────────────────────────────────────
        "timestamp":            timestamps,
        "data_id":              np.random.choice(data_ids,   n_records),
        "node_id":              np.random.choice(node_ids,   n_records),
        "data_type":            np.random.choice(DATA_TYPES, n_records),
        "gateway_id":           [f"gateway_{np.random.randint(1, NUM_CLOUD_GATEWAYS+1):02d}"
                                 for _ in range(n_records)],
        "region":               [f"region_{np.random.randint(1, 6)}"
                                 for _ in range(n_records)],

        # ── Access pattern (Poisson) ───────────────────────────────────────
        "access_frequency":     np.random.poisson(lam=5, size=n_records),

        # ── Latency & Recovery (Log-normal) ───────────────────────────────
        "latency_ms":           np.clip(
                                    np.random.lognormal(mean=3.5, sigma=0.8,
                                                        size=n_records),
                                    LATENCY_MIN_MS, LATENCY_MAX_MS).round(2),
        "recovery_time_ms":     np.clip(
                                    np.random.lognormal(mean=5.0, sigma=1.0,
                                                        size=n_records),
                                    50, 1000).round(2),

        # ── Bandwidth & Storage (Uniform) ──────────────────────────────────
        "bandwidth_mb_s":       np.random.uniform(10,  100, n_records).round(2),
        "storage_overhead_pct": np.random.uniform(5,   30,  n_records).round(2),

        # ── Spatial / size features ────────────────────────────────────────
        "data_size_mb":         np.random.exponential(scale=50,
                                                      size=n_records).round(2),
        "node_storage_tb":      np.random.uniform(STORAGE_MIN_TB,
                                                  STORAGE_MAX_TB,
                                                  n_records).round(2),

        # ── Redundancy & Cluster (SOM) ────────────────────────────────────
        "redundancy_factor":    np.random.randint(1, 4, n_records),
        "cluster_id":           np.random.randint(0,
                                                  SOM_GRID[0] * SOM_GRID[1],
                                                  n_records),

        # ── Node failure flag ─────────────────────────────────────────────
        "node_failed":          (np.random.uniform(0, 1, n_records) <
                                 np.random.uniform(FAILURE_PROB_MIN,
                                                   FAILURE_PROB_MAX,
                                                   n_records)).astype(int),
    })

    print(f"         Raw shape  : {df.shape}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# Step 2 – Preprocessing Pipeline (paper section 3.1)
# ─────────────────────────────────────────────────────────────────────────────

def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """
    1. Statistical outlier removal  (IQR method)
    2. Missing value imputation     (mean for numeric, mode for categorical)
    3. Min-max normalisation        (adds *_norm columns)
    """
    print("\n[Step 2] Preprocessing …")
    df = df.copy()

    numeric_cols = [
        "access_frequency", "latency_ms", "recovery_time_ms",
        "bandwidth_mb_s",   "storage_overhead_pct", "data_size_mb",
        "node_storage_tb"
    ]

    # 1. IQR-based outlier removal
    before = len(df)
    for col in numeric_cols:
        if col not in df.columns:
            continue
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR    = Q3 - Q1
        df     = df[(df[col] >= Q1 - 1.5 * IQR) &
                    (df[col] <= Q3 + 1.5 * IQR)]
    print(f"         Outlier removal : {before:,} → {len(df):,} rows")

    # 2. Imputation
    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].mean())
    for col in ["data_type", "node_id", "data_id", "gateway_id", "region"]:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].mode()[0])

    # 3. Min-max normalisation
    for col in numeric_cols:
        if col not in df.columns:
            continue
        col_min, col_max = df[col].min(), df[col].max()
        df[f"{col}_norm"] = (
            ((df[col] - col_min) / (col_max - col_min)).round(4)
            if col_max > col_min else 0.0
        )

    print(f"         Final shape    : {df.shape}")
    return df.reset_index(drop=True)


# ─────────────────────────────────────────────────────────────────────────────
# Step 3 – Feature Engineering (Priority Score & Backup Outcome)
# ─────────────────────────────────────────────────────────────────────────────

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds:
      priority_score  — Equation 1: Pij = β1·Aij + β2·Redundancy_j + β3·Storage_i
      backup_success  — derived outcome label
    """
    print("\n[Step 3] Feature engineering …")

    access_norm     = df["access_frequency"]     / df["access_frequency"].max()
    redundancy_norm = df["redundancy_factor"]     / 3
    storage_norm    = df["storage_overhead_pct"]  / 100

    df["priority_score"] = (
        BETA1 * access_norm +
        BETA2 * redundancy_norm +
        BETA3 * storage_norm
    ).round(4)

    df["backup_success"] = (
        (df["priority_score"] > 0.3) & (df["node_failed"] == 0)
    ).astype(int)

    success_rate = df["backup_success"].mean() * 100
    print(f"         Backup success rate : {success_rate:.2f}%")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# Step 4 – Validate Distributions (paper section 3.1)
# ─────────────────────────────────────────────────────────────────────────────

def validate(df: pd.DataFrame):
    print("\n[Step 4] Distribution Validation")
    print("─" * 50)

    checks = {
        "access_frequency  (Poisson)":     "access_frequency",
        "latency_ms        (Log-normal)":  "latency_ms",
        "recovery_time_ms  (Log-normal)":  "recovery_time_ms",
        "bandwidth_mb_s    (Uniform)":     "bandwidth_mb_s",
        "storage_overhead  (Uniform)":     "storage_overhead_pct",
    }
    for label, col in checks.items():
        s = df[col]
        print(f"\n  {label}")
        print(f"    mean={s.mean():.2f}  std={s.std():.2f}  "
              f"min={s.min():.2f}  max={s.max():.2f}")

    print(f"\n  Records       : {len(df):,}")
    print(f"  Backup success: {df['backup_success'].mean()*100:.2f}%")
    print(f"  Node failures : {df['node_failed'].mean()*100:.2f}%")
    print("─" * 50)


# ─────────────────────────────────────────────────────────────────────────────
# Step 5 – Save to Google Drive
# ─────────────────────────────────────────────────────────────────────────────

def save_to_drive(df: pd.DataFrame) -> str:
    full_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE)
    df.to_csv(full_path, index=False)
    size_kb = os.path.getsize(full_path) / 1024
    print(f"\n[Step 5] ✅ Dataset saved!")
    print(f"         Path    : {full_path}")
    print(f"         Rows    : {len(df):,}")
    print(f"         Columns : {len(df.columns)}")
    print(f"         Size    : {size_kb:,.1f} KB  ({size_kb/1024:.2f} MB)")
    return full_path


# ─────────────────────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────────────────────

def main() -> pd.DataFrame:
    print("=" * 55)
    print("  IoT Synthetic Traces – ADM-DE Dataset Generator")
    print("=" * 55 + "\n")

    raw_df   = generate_iot_synthetic_traces(NUM_RECORDS)
    clean_df = preprocess(raw_df)
    final_df = add_features(clean_df)
    validate(final_df)
    save_to_drive(final_df)

    print("\n── Column List ──")
    for i, col in enumerate(final_df.columns, 1):
        print(f"  {i:02d}. {col}")

    print("\n── Sample Rows ──")
    print(final_df[["timestamp", "data_id", "node_id", "data_type",
                     "access_frequency", "latency_ms",
                     "priority_score", "backup_success"]
                   ].head(5).to_string(index=False))

    return final_df


# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df = main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted successfully.
📁 Save path : /content/drive/MyDrive/Github/Parthiban/iot_synthetic_traces.csv

  IoT Synthetic Traces – ADM-DE Dataset Generator

[Step 1] Generating 10,000 raw IoT trace records …
         Raw shape  : (10000, 16)

[Step 2] Preprocessing …
         Outlier removal : 10,000 → 8,158 rows
         Final shape    : (8158, 23)

[Step 3] Feature engineering …
         Backup success rate : 81.18%

[Step 4] Distribution Validation
──────────────────────────────────────────────────

  access_frequency  (Poisson)
    mean=4.89  std=2.08  min=0.00  max=10.00

  latency_ms        (Log-normal)
    mean=37.65  std=24.64  min=5.00  max=114.29

  recovery_time_ms  (Log-normal)
    mean=176.34  std=132.00  min=50.00  max=600.45

  bandwidth_mb_s    (Uniform)
    mean=55.22  std=26.19  min=10.01  max=99.99

  storage_overhead  (Uniform)


MAWI dataset

In [6]:
import pandas as pd
import numpy as np
import random
import os
from datetime import datetime, timedelta

# -----------------------------
# SAVE PATH (your folder)
# -----------------------------
save_path = "/content/drive/MyDrive/Github/Parthiban"
os.makedirs(save_path, exist_ok=True)

file_path = os.path.join(save_path, "mawi_like_dataset.csv")

# -----------------------------
# PARAMETERS
# -----------------------------
NUM_RECORDS = 50000
NUM_NODES = 50
NUM_DATA_ITEMS = 1000

start_time = datetime(2025, 1, 1)

# -----------------------------
# FUNCTIONS
# -----------------------------
def generate_timestamp():
    return start_time + timedelta(seconds=random.randint(0, 86400*30))

def generate_protocol():
    return random.choice(['TCP', 'UDP', 'ICMP'])

def generate_data_type():
    return random.choice(['sensor', 'video', 'log', 'text', 'image'])

def generate_ip():
    return f"{random.randint(1,255)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,255)}"

# -----------------------------
# DATA GENERATION
# -----------------------------
data = []

for i in range(NUM_RECORDS):
    node_id = random.randint(1, NUM_NODES)
    data_id = random.randint(1, NUM_DATA_ITEMS)

    access_freq = np.random.poisson(lam=5)
    latency = np.random.lognormal(mean=2, sigma=0.5)
    bandwidth = random.uniform(1, 100)

    data.append({
        "timestamp": generate_timestamp(),
        "node_id": node_id,
        "data_id": data_id,
        "src_ip": generate_ip(),
        "dst_ip": generate_ip(),
        "protocol": generate_protocol(),
        "packet_size": random.randint(64, 1500),
        "access_frequency": access_freq,
        "latency_ms": latency,
        "bandwidth_MBps": bandwidth,
        "data_type": generate_data_type()
    })

df = pd.DataFrame(data)

# -----------------------------
# SAVE FILE
# -----------------------------
df.to_csv(file_path, index=False)

print("✅ Dataset saved at:", file_path)
print(df.head())

✅ Dataset saved at: /content/drive/MyDrive/Github/Parthiban/mawi_like_dataset.csv
            timestamp  node_id  data_id          src_ip           dst_ip  \
0 2025-01-06 22:09:43        5      665   244.67.129.99   128.119.31.225   
1 2025-01-27 14:58:16       36      772     178.33.8.99     212.4.23.202   
2 2025-01-30 20:07:31       20      908   109.193.75.20  226.167.197.201   
3 2025-01-07 09:31:33       13      934   24.66.211.159   83.117.206.100   
4 2025-01-24 13:13:45       47      120  131.31.230.245   203.156.202.69   

  protocol  packet_size  access_frequency  latency_ms  bandwidth_MBps  \
0     ICMP         1282                 8    4.240344       84.714486   
1      UDP          633                 4    4.566824       45.369202   
2     ICMP         1484                 5   10.029702       99.517404   
3     ICMP          389                 3    5.942816        7.506307   
4      UDP          551                 5    2.318013       39.034245   

  data_type  
0     im